# Myllia: Hybrid Multi-Head Bilinear + Residual Basis (h5ad gene embeddings)

This notebook:
- Reproduces `training_data_means.csv` from `training_cells.h5ad` (exact normalization)
- Builds a **gene embedding bank** from all 19,226 genes using TruncatedSVD on normalized single-cell data
- Trains a **hybrid model**: multi-head bilinear + residual basis, metric-aligned proxy loss
- Optionally runs pert-level CV to tune shrinkage
- Writes a submission CSV


In [1]:

import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad

import torch
import torch.nn as nn
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

from myllia_metric import myllia_score

print("torch:", torch.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

torch: 2.5.1+cu121
device: cuda


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

H5AD_PATH = "Data/training_cells.h5ad"
MEANS_PATH = "Data/training_data_means.csv"
VALMAP_PATH = "Data/pert_ids_val.csv"
SAMPLE_SUB_PATH = "Data/sample_submission.csv"

# embedding bank cache
EMB_CACHE = "Data/gene_embedding_bank_svd.npz"

# embedding + model dims
EMB_DIM = 128
H_HEADS = 4
RANK_R = 32
K_RES = 32

# training
MODEL_SEEDS = [6, 7, 8]
EPOCHS = 400
BATCH_PERTS = 16
LR = 2e-3
WD = 1e-4
DROPOUT = 0.10
EVAL_EVERY = 25
PATIENCE = 12

# metric-aligned gate (match official)
GATE_A = 0.0
GATE_B = 0.2
EPS = 1e-12

# loss weights
COS_BETA = 0.20  # tune (0.05..0.30)

# regularization weights
UDELTA_L2 = 1e-4
SCALE_L2  = 1e-4
BIAS_L2   = 1e-5
BASIS_L2  = 1e-5

# shrinkage alpha (will tune via CV cell below)
ALPHA_SHRINK = 0.92

print("EMB_DIM", EMB_DIM, "H_HEADS", H_HEADS, "RANK_R", RANK_R, "K_RES", K_RES)

EMB_DIM 128 H_HEADS 4 RANK_R 32 K_RES 32


## Load `training_data_means.csv` and build training deltas

In [3]:
df_means = pd.read_csv(MEANS_PATH)
gene_cols = [c for c in df_means.columns if c != "pert_symbol"]

baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_cols].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_perts = df_train["pert_symbol"].astype(str).to_numpy()

X_train_means = df_train[gene_cols].to_numpy(np.float32)
D_train = X_train_means - x_base[None, :]  # (80, 5127) deltas vs baseline

delta_baseline = D_train.mean(axis=0).astype(np.float32)

print("[means] rows:", len(df_means), "train perts:", len(train_perts), "genes:", len(gene_cols))
print("D_train:", D_train.shape)

[means] rows: 81 train perts: 80 genes: 5127
D_train: (80, 5127)


## Load `training_cells.h5ad` and verify the normalization matches exactly

In [4]:
adata = ad.read_h5ad(H5AD_PATH)

print("[h5ad] X shape:", adata.X.shape)
print("[h5ad] obs cols:", list(adata.obs.columns))
print("[h5ad] first var_names:", adata.var_names[:5].tolist())

# pick perturbation column
pert_col = None
for c in ["sgrna_symbol", "pert_symbol", "pert", "perturbation", "gene", "target_gene"]:
    if c in adata.obs.columns:
        pert_col = c
        break
if pert_col is None:
    raise ValueError("Could not find perturbation column in adata.obs. Set pert_col manually.")

print("[h5ad] using pert_col:", pert_col)

# counts matrix
X = adata.X
if not sp.issparse(X):
    X = sp.csr_matrix(X)
else:
    X = X.tocsr()

# normalize ALL genes: CPM10K then log2(1+x)
cell_sum = np.asarray(X.sum(axis=1)).ravel().astype(np.float64)
scale = (10000.0 / cell_sum).astype(np.float64)

Xn = X.multiply(scale[:, None]).tocsr()
Xn.data = np.log1p(Xn.data) / np.log(2.0)

# subset to 5127 genes in the exact order of training_data_means.csv
var = pd.Index(adata.var_names.astype(str))
idx = var.get_indexer(gene_cols)
if np.any(idx == -1):
    missing = [gene_cols[i] for i in np.where(idx == -1)[0]]
    print("missing genes:", len(missing), "first 40:", missing[:40])
    raise ValueError("h5ad gene names do not contain all 5127 challenge genes. Fix naming before proceeding.")

X5127 = Xn[:, idx]

# build per-pert mean across ALL genes
perts = adata.obs[pert_col].astype(str).to_numpy()
ser = pd.Series(perts)
groups = ser.groupby(ser).indices

names = []
means_all = []
for p, rows in groups.items():
    m = Xn[rows].mean(axis=0)  # 1 x 19226
    if sp.issparse(m):
        m = m.toarray()
    means_all.append(np.asarray(m).ravel().astype(np.float32))
    names.append(p)

means_all = np.vstack(means_all)  # (n_perts_in_h5ad, 19226)
names = np.array(names, dtype=object)

# baseline and training perts
base_idx = np.where(names == "non-targeting")[0]
if len(base_idx) != 1:
    raise ValueError("Expected exactly one non-targeting row in h5ad means")
x0 = means_all[base_idx[0]]  # (19226,)

# map h5ad perts -> rows
row_by_pert = {names[i]: i for i in range(len(names))}

# only the 80 training perts (from training_data_means.csv)
train_rows = []
missing_train_in_h5ad = []
for p in train_perts:
    if p not in row_by_pert:
        missing_train_in_h5ad.append(p)
    else:
        train_rows.append(row_by_pert[p])

if len(missing_train_in_h5ad) > 0:
    print("missing train perts in h5ad:", missing_train_in_h5ad)
    raise ValueError("Train perts missing in h5ad, cannot proceed")

M_train = means_all[train_rows]              # (80, 19226)
D_full  = (M_train - x0[None, :]).astype(np.float32)  # (80, 19226)

print("D_full:", D_full.shape)

[h5ad] X shape: (17882, 19226)
[h5ad] obs cols: ['nCount_RNA', 'nFeature_RNA', 'percent.mt', 'sgrna_id', 'sgrna_symbol', 'channel']
[h5ad] first var_names: ['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A3GALT2']
[h5ad] using pert_col: sgrna_symbol
D_full: (80, 19226)


## Build gene embedding bank from all cells (19,226 genes)

We compute TruncatedSVD on the normalized cell matrix `Xn` and use the right singular vectors as gene embeddings.
This gives embeddings for **all genes** in `adata.var_names`, so perturbations not in the 5,127 output list still get embeddings.

In [6]:
EMB_DIM = 80
svd = TruncatedSVD(n_components=EMB_DIM, random_state=6)
svd.fit(D_full.T)  # genes are samples, perts are features

gene_emb = svd.transform(D_full.T).astype(np.float32)  # (19226, EMB_DIM)

# unit normalize embeddings
gene_emb = gene_emb / (np.linalg.norm(gene_emb, axis=1, keepdims=True) + 1e-12)

genes_all = adata.var_names.astype(str).to_numpy()
gene2vec = {genes_all[i].upper(): gene_emb[i] for i in range(len(genes_all))}

# build Z_train and U_out
Z_train = np.vstack([gene2vec[g.upper()] for g in train_perts]).astype(np.float32)
U_out   = np.vstack([gene2vec[g.upper()] for g in gene_cols]).astype(np.float32)

print("Z_train:", Z_train.shape, "U_out:", U_out.shape)

Z_train: (80, 80) U_out: (5127, 80)


## Metric wrapper and shrinkage helper

In [7]:
def score_delta(dt, dp):
    dt = dt.astype(np.float32, copy=False)
    dp = dp.astype(np.float32, copy=False)
    r = myllia_score(dt, dp)
    return {
        "score": float(r.score),
        "wcos": float(r.wcos),
        "mean_term": float(r.mean_term),
        "pred_wmae": float(r.pred_wmae),
    }

def apply_shrink(pred, baseline, alpha):
    return alpha * pred + (1.0 - alpha) * baseline[None, :]

base_pred = np.tile(delta_baseline[None, :], (D_train.shape[0], 1))
print("baseline score (train-as-val):", score_delta(D_train, base_pred))

baseline score (train-as-val): {'score': 0.08682197230711175, 'wcos': 0.4714401364326477, 'mean_term': 0.18416330218315125, 'pred_wmae': 0.0809405967593193}


## Model: Multi-Head Bilinear + Residual Basis

In [8]:
# Torch tensors
Y = D_train.astype(np.float32)
N, G = Y.shape

Zt = torch.tensor(Z_train, device=device)          # (N, d)
Uo = torch.tensor(U_out, device=device)            # (G, d)
Yt = torch.tensor(Y, device=device)                # (N, G)
bias_init_t = torch.tensor(delta_baseline, device=device)  # (G,)

def gate_smoothstep(x, a=GATE_A, b=GATE_B):
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def weighted_l1_like(delta_true, delta_pred, eps=EPS):
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)
    err = torch.abs(delta_pred - delta_true)
    num = torch.sum(w * err, dim=1)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)
    return torch.mean(num / den)

def weighted_cosine_loss(delta_true, delta_pred, eps=EPS):
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)
    dtw = w * delta_true
    dpw = w * delta_pred
    num = torch.sum(dtw * dpw, dim=1)
    den = torch.sqrt(torch.sum(dtw * dtw, dim=1) * torch.sum(dpw * dpw, dim=1) + eps)
    cos = num / den
    return torch.mean(1.0 - cos)

class HybridMH(nn.Module):
    def __init__(self, d, H, r, k_res, dropout, G, bias_init):
        super().__init__()
        self.H = H
        self.r = r
        self.k_res = k_res

        self.proj_p = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d, 2*r),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(2*r, r),
                nn.LayerNorm(r),
            ) for _ in range(H)
        ])
        self.proj_o = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d, 2*r),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(2*r, r),
                nn.LayerNorm(r),
            ) for _ in range(H)
        ])

        self.coef = nn.Sequential(
            nn.Linear(d, 2*k_res),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2*k_res, k_res),
        )
        self.B = nn.Parameter(torch.zeros(G, k_res))
        nn.init.normal_(self.B, mean=0.0, std=0.02)

        self.u_out_delta = nn.Parameter(torch.zeros(G, d))

        self.scale_gene = nn.Parameter(torch.ones(G))
        self.bias_gene  = nn.Parameter(bias_init.clone())
        self.bias_global = nn.Parameter(torch.zeros(1))

    def forward(self, z_pert, u_out_base):
        u = u_out_base + self.u_out_delta

        y = 0.0
        for h in range(self.H):
            p = self.proj_p[h](z_pert)
            o = self.proj_o[h](u)
            y = y + (p @ o.T)

        c = self.coef(z_pert)
        y = y + (c @ self.B.T)

        y = y * self.scale_gene[None, :]
        y = y + self.bias_gene[None, :] + self.bias_global
        return y

def total_loss(model, dt, dp):
    l1 = weighted_l1_like(dt, dp)
    lc = weighted_cosine_loss(dt, dp)

    reg_u = torch.mean(model.u_out_delta * model.u_out_delta)
    reg_s = torch.mean((model.scale_gene - 1.0) ** 2)
    reg_b = torch.mean((model.bias_gene - bias_init_t) ** 2)
    reg_B = torch.mean(model.B * model.B)

    return l1 + COS_BETA * lc + UDELTA_L2*reg_u + SCALE_L2*reg_s + BIAS_L2*reg_b + BASIS_L2*reg_B

print("N", N, "G", G)

N 80 G 5127


## Optional: CV to tune shrinkage alpha (recommended)

This runs 8-fold CV over perturbations and picks the best alpha per fold, then sets `ALPHA_SHRINK` to the median.
Set `RUN_CV = True` to run.

In [12]:
def train_fold(tr_idx, va_idx, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = HybridMH(
        d=Zt.shape[1], H=H_HEADS, r=RANK_R, k_res=K_RES,
        dropout=DROPOUT, G=G, bias_init=bias_init_t
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    best_score = -1e18
    best_state = None
    patience = 0

    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = np.asarray(tr_idx).copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_PERTS):
            b = perm[start:start + BATCH_PERTS]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo)
            loss = total_loss(model, Yt.index_select(0, b_t), pred)

            opt.zero_grad()
            loss.backward()
            opt.step()

        sched.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(Zt.index_select(0, va_idx_t), Uo).detach().cpu().numpy().astype(np.float32)

            va_true = Y[va_idx]

            best_a = 1.0
            best_sc = -1e18
            for a in np.linspace(-2, 0.30, 30):
                pred_a = apply_shrink(va_pred, delta_baseline, float(a))
                sc = score_delta(va_true, pred_a)["score"]
                if sc > best_sc:
                    best_sc = sc
                    best_a = float(a)

            if best_sc > best_score:
                best_score = best_sc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_a, best_state

RUN_CV = True

if RUN_CV:
    kf = KFold(n_splits=8, shuffle=True, random_state=6)
    fold_scores = []
    fold_alphas = []
    for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(N)), 1):
        sc, a, _ = train_fold(tr_idx, va_idx, seed=MODEL_SEEDS[0])
        fold_scores.append(sc)
        fold_alphas.append(a)
        print(f"fold {fold}: best_score={sc:.6f} best_alpha={a:.3f}")

    print("cv mean:", float(np.mean(fold_scores)), "std:", float(np.std(fold_scores)))
    ALPHA_SHRINK = float(np.median(fold_alphas))
    print("set ALPHA_SHRINK =", ALPHA_SHRINK)
else:
    print("RUN_CV is False. Using ALPHA_SHRINK =", ALPHA_SHRINK)

fold 1: best_score=0.100774 best_alpha=-0.097
fold 2: best_score=0.074487 best_alpha=-0.097
fold 3: best_score=0.070578 best_alpha=0.062
fold 4: best_score=0.078405 best_alpha=-0.017
fold 5: best_score=0.118536 best_alpha=0.221
fold 6: best_score=0.137668 best_alpha=-0.097
fold 7: best_score=0.123090 best_alpha=0.062
fold 8: best_score=0.087356 best_alpha=0.062
cv mean: 0.09886178473088186 std: 0.023528268917498183
set ALPHA_SHRINK = 0.022413793103448154


fold 1: best_score=0.023921 best_alpha=0.700
fold 2: best_score=0.019547 best_alpha=0.700
fold 3: best_score=0.014332 best_alpha=0.700
fold 4: best_score=0.017164 best_alpha=0.700
fold 5: best_score=0.040813 best_alpha=0.700
fold 6: best_score=0.043116 best_alpha=0.700
fold 7: best_score=0.031240 best_alpha=0.700
fold 8: best_score=0.014287 best_alpha=0.700
cv mean: 0.025552606627948254 std: 0.010814576120829615
set ALPHA_SHRINK = 0.7

## Train full models (multi-seed ensemble) and write submission

In [13]:
def fit_full(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = HybridMH(
        d=Zt.shape[1], H=H_HEADS, r=RANK_R, k_res=K_RES,
        dropout=DROPOUT, G=G, bias_init=bias_init_t
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    best_score = -1e18
    best_state = None
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = np.arange(N)
        np.random.shuffle(perm)

        for start in range(0, N, BATCH_PERTS):
            b = perm[start:start + BATCH_PERTS]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo)
            loss = total_loss(model, Yt.index_select(0, b_t), pred)

            opt.zero_grad()
            loss.backward()
            opt.step()

        sched.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                pred_np = model(Zt, Uo).detach().cpu().numpy().astype(np.float32)

            pred_np = apply_shrink(pred_np, delta_baseline, ALPHA_SHRINK)
            s = score_delta(Y, pred_np)
            sc = s["score"]
            print(f"[seed {seed}] epoch={epoch:4d} train_score={sc:.6f} wcos={s['wcos']:.6f} pred_wmae={s['pred_wmae']:.6f}")

            if sc > best_score:
                best_score = sc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model

models = [fit_full(sd) for sd in MODEL_SEEDS]
print("Trained models:", len(models))

def predict_delta_for_pert(pert_gene):
    g = str(pert_gene).upper()
    if g not in gene2vec:
        return delta_baseline.copy()

    z = torch.tensor(gene2vec[g][None, :].astype(np.float32), device=device)
    preds = []
    with torch.no_grad():
        for m in models:
            y = m(z, Uo).detach().cpu().numpy().astype(np.float32)[0]
            preds.append(y)

    yhat = np.mean(np.stack(preds, axis=0), axis=0).astype(np.float32)
    yhat = ALPHA_SHRINK * yhat + (1.0 - ALPHA_SHRINK) * delta_baseline
    return yhat

df_valmap = pd.read_csv(VALMAP_PATH)
val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))

sub = pd.read_csv(SAMPLE_SUB_PATH)
sub["pert_id"] = sub["pert_id"].astype(str)
sub_gene_cols = [c for c in sub.columns if c != "pert_id"]

idx = {g: i for i, g in enumerate(gene_cols)}
perm = [idx[g] for g in sub_gene_cols]

sub.loc[:, sub_gene_cols] = np.tile(delta_baseline[perm][None, :], (len(sub), 1))

hit = 0
for pid, gene in val_map.items():
    vec = predict_delta_for_pert(gene)[perm]
    m = (sub["pert_id"] == str(pid))
    if m.any():
        sub.loc[m, sub_gene_cols] = vec[None, :]
        hit += int(m.sum())

out_path = "submission_hybrid_mh_residual.csv"
sub.to_csv(out_path, index=False)
print("Wrote:", out_path, "| filled:", hit)

[seed 6] epoch=  25 train_score=0.116981 wcos=0.514956 pred_wmae=0.079187
[seed 6] epoch=  50 train_score=0.148101 wcos=0.592124 pred_wmae=0.078278
[seed 6] epoch=  75 train_score=0.142415 wcos=0.589278 pred_wmae=0.078631
[seed 6] epoch= 100 train_score=0.140341 wcos=0.588230 pred_wmae=0.078709
[seed 6] epoch= 125 train_score=0.132726 wcos=0.568600 pred_wmae=0.078870
[seed 6] epoch= 150 train_score=0.141717 wcos=0.582038 pred_wmae=0.078413
[seed 6] epoch= 175 train_score=0.128997 wcos=0.563638 pred_wmae=0.079080
[seed 6] epoch= 200 train_score=0.123799 wcos=0.553671 pred_wmae=0.079286
[seed 6] epoch= 225 train_score=0.124380 wcos=0.552935 pred_wmae=0.079199
[seed 6] epoch= 250 train_score=0.121939 wcos=0.548719 pred_wmae=0.079296
[seed 6] epoch= 275 train_score=0.120797 wcos=0.546722 pred_wmae=0.079349
[seed 6] epoch= 300 train_score=0.120829 wcos=0.546142 pred_wmae=0.079334
[seed 6] epoch= 325 train_score=0.120730 wcos=0.546005 pred_wmae=0.079334
[seed 6] epoch= 350 train_score=0.1207

KeyboardInterrupt: 

[seed 6] epoch=  25 train_score=0.000000 wcos=0.341268 pred_wmae=0.345794
[seed 6] epoch=  50 train_score=0.062496 wcos=0.749689 pred_wmae=0.154856
[seed 6] epoch=  75 train_score=0.059306 wcos=0.769081 pred_wmae=0.160722
[seed 6] epoch= 100 train_score=0.048632 wcos=0.690838 pred_wmae=0.180086
[seed 6] epoch= 125 train_score=0.258332 wcos=0.839284 pred_wmae=0.108974
[seed 6] epoch= 150 train_score=0.355655 wcos=0.856620 pred_wmae=0.093074
[seed 6] epoch= 175 train_score=0.229752 wcos=0.823118 pred_wmae=0.108586
[seed 6] epoch= 200 train_score=0.864994 wcos=0.931711 pred_wmae=0.051142
[seed 6] epoch= 225 train_score=1.069900 wcos=0.947153 pred_wmae=0.044170
[seed 6] epoch= 250 train_score=1.280484 wcos=0.967441 pred_wmae=0.037075
[seed 6] epoch= 275 train_score=1.414764 wcos=0.982252 pred_wmae=0.034511
[seed 6] epoch= 300 train_score=1.392321 wcos=0.982582 pred_wmae=0.034977
[seed 6] epoch= 325 train_score=1.431653 wcos=0.984325 pred_wmae=0.034051
[seed 6] epoch= 350 train_score=1.435244 wcos=0.984665 pred_wmae=0.033985
[seed 6] epoch= 375 train_score=1.436536 wcos=0.985246 pred_wmae=0.033965
[seed 6] epoch= 400 train_score=1.436025 wcos=0.985429 pred_wmae=0.033990
[seed 7] epoch=  25 train_score=0.000000 wcos=0.442398 pred_wmae=0.477607
[seed 7] epoch=  50 train_score=0.002483 wcos=0.445681 pred_wmae=0.287584
[seed 7] epoch=  75 train_score=0.146782 wcos=0.808785 pred_wmae=0.133376
[seed 7] epoch= 100 train_score=0.003221 wcos=0.465324 pred_wmae=0.285401
[seed 7] epoch= 125 train_score=0.524167 wcos=0.899180 pred_wmae=0.068713
[seed 7] epoch= 150 train_score=0.036932 wcos=0.756127 pred_wmae=0.131665
[seed 7] epoch= 175 train_score=1.198658 wcos=0.960267 pred_wmae=0.039228
[seed 7] epoch= 200 train_score=0.600230 wcos=0.892866 pred_wmae=0.060372
[seed 7] epoch= 225 train_score=1.338952 wcos=0.975479 pred_wmae=0.035995
...
[seed 8] epoch= 375 train_score=1.428714 wcos=0.984861 pred_wmae=0.034196
[seed 8] epoch= 400 train_score=1.428618 wcos=0.984948 pred_wmae=0.034197
Trained models: 3
Wrote: submission_hybrid_mh_residual.csv | filled: 60
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...

In [14]:
missing = ['BRD4','CHD4','DNAJA3','INO80','KAT8','KDM4A','PMEL','SETD1A']
print([g for g in missing if g not in set(gene_cols)])

['BRD4', 'CHD4', 'DNAJA3', 'INO80', 'KAT8', 'KDM4A', 'PMEL', 'SETD1A']


In [15]:
gene_cols

['A1BG',
 'A1CF',
 'AADAC',
 'AAK1',
 'AARS1',
 'AASS',
 'ABCA1',
 'ABCA12',
 'ABCA5',
 'ABCB5',
 'ABCC1',
 'ABCC2',
 'ABCC3',
 'ABCC4',
 'ABCG2',
 'ABHD12B',
 'ABHD17C',
 'ABHD2',
 'ABHD3',
 'ABHD4',
 'ABI3BP',
 'ABLIM1',
 'ABLIM3',
 'ABRACL',
 'ABTB2',
 'ABTB3',
 'ACAD8',
 'ACADVL',
 'ACAT2',
 'ACER2',
 'ACKR3',
 'ACLY',
 'ACO1',
 'ACOT13',
 'ACOT2',
 'ACOX2',
 'ACP6',
 'ACSBG1',
 'ACSL4',
 'ACSM2A',
 'ACSS3',
 'ACTA2',
 'ACTB',
 'ACTG1',
 'ACTG2',
 'ACTL6A',
 'ACTL8',
 'ACTN1',
 'ACTN4',
 'ACTR2',
 'ACTR3',
 'ACVR1B',
 'ACYP2',
 'ADAM10',
 'ADAM12',
 'ADAM19',
 'ADAM21',
 'ADAM22',
 'ADAM23',
 'ADAM32',
 'ADAM9',
 'ADAMTS10',
 'ADAMTS12',
 'ADAMTS13',
 'ADAMTS16',
 'ADAMTS18',
 'ADAMTS2',
 'ADAMTS20',
 'ADAMTS6',
 'ADAMTS7',
 'ADAMTS8',
 'ADAMTS9',
 'ADAMTSL3',
 'ADAP2',
 'ADARB1',
 'ADCY9',
 'ADD2',
 'ADD3',
 'ADGRB3',
 'ADGRE1',
 'ADGRE3',
 'ADGRE5',
 'ADGRG1',
 'ADGRG6',
 'ADGRL3',
 'ADGRL4',
 'ADI1',
 'ADISSP',
 'ADK',
 'ADM',
 'ADM2',
 'ADM5',
 'ADORA1',
 'ADORA2B',
 'ADRB2',
 